In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/evelynuntariady/thesis/data_thesis_final.csv
/kaggle/input/datasets/evelynuntariady/thesis/raw_data_all.csv
/kaggle/input/datasets/evelynuntariady/thesis/thesis_full.csv
/kaggle/input/datasets/evelynuntariady/thesis/data_thesis_majority_vote.csv


In [2]:
! pip install -q optuna
! pip install -q emoji

In [3]:
import pandas as pd
import numpy as np
import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from huggingface_hub import login
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
import optuna
import ast
import re
import emoji
from transformers.utils import logging as hf_logging
import gc
import shutil
import random
import math
from transformers import TrainingArguments, Trainer


In [4]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [5]:
# df_test = pd.read_csv("/kaggle/input/datasets/evelynuntariady/thesis/data_thesis_final.csv")

In [6]:
df_main = pd.read_csv('/kaggle/input/datasets/evelynuntariady/thesis/thesis_full.csv')

In [7]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27833 entries, 0 to 27832
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   text_id                        27833 non-null  object
 1   annotators_id                  27833 non-null  object
 2   text                           27832 non-null  object
 3   topic                          27833 non-null  object
 4   is_noise_or_spam_text          27833 non-null  object
 5   toxicity                       27833 non-null  object
 6   polarized                      27833 non-null  object
 7   profanity_obscenity            27833 non-null  object
 8   threat_incitement_to_violence  27833 non-null  object
 9   insults                        27833 non-null  object
 10  identity_attack                27833 non-null  object
 11  sexually_explicit              27833 non-null  object
 12  key                            27833 non-null  object
 13  c

In [8]:
# df_test.info()

In [9]:
# temp = df_test[df_test[['text_id', 'annotators_id']].duplicated(keep=False)]
# temp

In [10]:
cols = [
'is_noise_or_spam_text',
'toxicity',
'polarized',
'profanity_obscenity',
'threat_incitement_to_violence',
'insults',
'identity_attack',
'sexually_explicit'
]

for col in cols:
    df_main[col] = df_main[col].apply(lambda x: ast.literal_eval(x) if isinstance(x,str) else x)

In [11]:
print(df_main['is_noise_or_spam_text'].iloc[0])
print(type(df_main['is_noise_or_spam_text'].iloc[0]))

print(df_main['polarized'].iloc[0])
print(type(df_main['polarized'].iloc[0]))

print(df_main['profanity_obscenity'].iloc[0])
print(type(df_main['profanity_obscenity'].iloc[0]))

print(df_main['insults'].iloc[0])
print(type(df_main['insults'].iloc[0]))

print(df_main['sexually_explicit'].iloc[0])
print(type(df_main['sexually_explicit'].iloc[0]))

print(df_main['identity_attack'].iloc[0])
print(type(df_main['identity_attack'].iloc[0]))

print(df_main['toxicity'].iloc[0])
print(type(df_main['toxicity'].iloc[0]))

['0', '0']
<class 'list'>
['0', '0']
<class 'list'>
['0', '0']
<class 'list'>
['0', '0']
<class 'list'>
['0', '0']
<class 'list'>
['0', '0']
<class 'list'>
['0', '0']
<class 'list'>


In [12]:
df_main.iloc[5389]

text_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           1-6084
annotators_id                                                                                                                                                                                                                                                  

In [13]:
df_main.at[5389, 'is_noise_or_spam_text'] = [0, 1, 0]

In [14]:
for col in cols:
    for idx, x in df_main[col].items():
        try:
            for i in x:
                int(i)
        except Exception as e:
            print(f"Column: {col}")
            print(f"Row: {idx}")
            print("Cell:", x)
            print("Cell type:", type(x))
            print("Element types:", [type(i) for i in x])
            raise

### MAJORITY VOTE WO POLARIZED

In [ ]:
print(cols)
cols2 = [c for c in cols if c != 'polarized']
print(cols2)

['is_noise_or_spam_text', 'toxicity', 'polarized', 'profanity_obscenity', 'threat_incitement_to_violence', 'insults', 'identity_attack', 'sexually_explicit']
['is_noise_or_spam_text', 'toxicity', 'profanity_obscenity', 'threat_incitement_to_violence', 'insults', 'identity_attack', 'sexually_explicit']


In [ ]:
def majority_vote(x):
    x = [int(i) for i in x]
    return round(sum(x)/len(x))

for col in cols2:
    df_main[col] = df_main[col].apply(majority_vote)

In [17]:
# df_main = df.copy()
df_main = df_main[df_main['is_noise_or_spam_text'] == 0]
df_main = df_main.dropna()
df_main.isnull().sum()

text_id                          0
annotators_id                    0
text                             0
topic                            0
is_noise_or_spam_text            0
toxicity                         0
polarized                        0
profanity_obscenity              0
threat_incitement_to_violence    0
insults                          0
identity_attack                  0
sexually_explicit                0
key                              0
cleaned_text                     0
processed_text                   0
dtype: int64

In [18]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26375 entries, 0 to 27832
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   text_id                        26375 non-null  object
 1   annotators_id                  26375 non-null  object
 2   text                           26375 non-null  object
 3   topic                          26375 non-null  object
 4   is_noise_or_spam_text          26375 non-null  int64 
 5   toxicity                       26375 non-null  int64 
 6   polarized                      26375 non-null  object
 7   profanity_obscenity            26375 non-null  int64 
 8   threat_incitement_to_violence  26375 non-null  int64 
 9   insults                        26375 non-null  int64 
 10  identity_attack                26375 non-null  int64 
 11  sexually_explicit              26375 non-null  int64 
 12  key                            26375 non-null  object
 13  cleane

In [19]:
df_main.head(1)

,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,key,cleaned_text,processed_text
0,1-1,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan Kondisi di Gaza 2023,UNKNOWN,0,0,"[0, 0]",0,0,0,0,0,"1-1_['7', '15']",Wajah jurnalis dibalut perban saat melaporkan kondisi di Gaza 2023,wajah jurnalis dibalut perban saat melaporkan kondisi di gaza


### PROCESS POLARIZED USING AGG (LIKE PAPER)

In [20]:
x = df_main["polarized"].iloc[0]
print(type(x), repr(x))

<class 'list'> ['0', '0']


In [ ]:
import numpy as np

df_main["polarization_list"] = df_main["polarized"].apply(lambda xs: [int(v) for v in xs])
df_main["pol_avg"] = df_main["polarization_list"].apply(lambda xs: float(np.mean(xs)))

print(df_main[["polarized", "pol_avg"]].head())

   polarized   pol_avg
0     [0, 0]  0.000000
1     [1, 1]  1.000000
2  [0, 0, 0]  0.000000
3  [1, 0, 0]  0.333333
4     [1, 1]  1.000000


In [22]:
df_main.head(5)

,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,key,cleaned_text,processed_text,polarization_list,pol_avg
0,1-1,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan Kondisi di Gaza 2023,UNKNOWN,0,0,"[0, 0]",0,0,0,0,0,"1-1_['7', '15']",Wajah jurnalis dibalut perban saat melaporkan kondisi di Gaza 2023,wajah jurnalis dibalut perban saat melaporkan kondisi di gaza,"[0, 0]",0.000000
1,1-10,"['7', '15']","Elektabilitas Paslon 02 sebentar lagi terjun bebas gara2 Gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, Ijazah dan tidak hadir",Disabilitas,0,0,"[1, 1]",0,0,0,0,0,"1-10_['7', '15']","Elektabilitas pasangan calon 02 sebentar lagi terjun bebas gara-gara Gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, ijazah, dan tidak hadir","elektabilitas pasangan calon sebentar lagi terjun bebas garagara gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, ijazah, dan tidak hadir","[1, 1]",1.000000
2,1-100,"['7', '15', '32']","Gini aja deh @KPU_ID usul aja nih. Gimana kalo misalnya gak usah ada debat aja kalo ga boleh dikritik ? Diubah jadi lomba joget siapa yang paling GEndut SeMpOYongan 😂🤦👎 Menarik gak ? Atur lah, kalian kan penguasa.. 😁",Disabilitas,0,1,"[0, 0, 0]",0,0,1,0,0,"1-100_['7', '15', '32']","Gini aja deh @KPU_ID, usul aja nih. Gimana kalau misalnya tidak usah ada debat kalau nggak boleh dikritik? Diubah jadi lomba joget siapa yang paling gendut sempoyongan 😂🤦👎 Menarik nggak? Atur lah, kalian kan penguasa.. 😁","gini saja deh , usul saja nih. gimana kalau misalnya tidak usah ada debat kalau tidak boleh dikritik? diubah jadi lomba joget siapa yang paling gendut sempoyongan menarik nggak? atur lah, kalian kan penguasa..","[0, 0, 0]",0.000000
3,1-1000,"['7', '15', '32']",Relawan Tionghoa Kalbar yg di Jakarta mendeklarasikan dukungan kepada Ganjar Pranowo. Mario Dandy Bu D Megumi Putri Ariani Mamat Cilacap Mahkamah Konstitusi,Tionghoa,0,0,"[1, 0, 0]",0,0,0,0,0,"1-1000_['7', '15', '32']","Relawan Tionghoa Kalbar yang di Jakarta mendeklarasikan dukungan kepada Ganjar Pranowo. Nama terkait: Mario Dandy, Bu D, Megumi, Putri Ariani, Mamat. Cilacap. Mahkamah Konstitusi.","relawan tionghoa kalbar yang di jakarta mendeklarasikan dukungan kepada ganjar pranowo. nama terkait mario dandy, bu d, megumi, putri ariani, mamat. cilacap. mahkamah konstitusi.","[1, 0, 0]",0.333333
4,1-1001,"['10', '18']","Kristen, Hindu, Islam dapat perlakuan istimewa dari pak Anies Ncep ketar-ketir",Kristen,0,0,"[1, 1]",0,0,0,0,0,"1-1001_['10', '18']","Kristen, Hindu, Islam dapat perlakuan istimewa dari Pak Anies Ncep, ketar-ketir","kristen, hindu, islam dapat perlakuan istimewa dari pak anies ncep, ketarketir","[1, 1]",1.000000


## PREPROCESSING

In [23]:
def preprocess_tweet(text):
    if not isinstance(text, str):
        return ""

    text = emoji.demojize(text).lower()
     # 1. remove username (@user)
    text = re.sub(r'@\w+', '', text)

    # 2. remove URL
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3. remove numbers
    #    Catatan: ini juga menghapus angka yang informatif di konteks politik (mis. 'paslon 02')
    text = re.sub(r'\d+', '', text)

    # 4. remove unwanted symbols (keep .,!?)
    text = re.sub(r'[^\w\s.,!?]', '', text)

    # 6. normalize repeated chars (contoh: anjiiing → anjiing)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 7. lowercase (optional)
    text = text.lower()

    # 8. remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [24]:
from tqdm import tqdm
tqdm.pandas()

df_main["processed_text"] = (
    df_main["cleaned_text"]
    .fillna("")
    .astype(str)
    .progress_apply(
        lambda x: preprocess_tweet(x)
    )
)

100%|██████████| 26375/26375 [00:06<00:00, 3904.44it/s]


In [25]:
df_main['toxicity'].value_counts()

toxicity
0    23976
1     2399
Name: count, dtype: int64

In [26]:
df_main.head(2)

,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,key,cleaned_text,processed_text,polarization_list,pol_avg
0,1-1,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan Kondisi di Gaza 2023,UNKNOWN,0,0,"[0, 0]",0,0,0,0,0,"1-1_['7', '15']",Wajah jurnalis dibalut perban saat melaporkan kondisi di Gaza 2023,wajah jurnalis dibalut perban saat melaporkan kondisi di gaza,"[0, 0]",0.0
1,1-10,"['7', '15']","Elektabilitas Paslon 02 sebentar lagi terjun bebas gara2 Gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, Ijazah dan tidak hadir",Disabilitas,0,0,"[1, 1]",0,0,0,0,0,"1-10_['7', '15']","Elektabilitas pasangan calon 02 sebentar lagi terjun bebas gara-gara Gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, ijazah, dan tidak hadir","elektabilitas pasangan calon sebentar lagi terjun bebas garagara gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, ijazah, dan tidak hadir","[1, 1]",1.0


# TRY BALANCING 1:3 RATIO (LIKE PAPER) -- UNDERSAMPLING

In [ ]:
pos = df_main[df_main["toxicity"] == 1]
neg = df_main[df_main["toxicity"] == 0].sample(n=3 * len(pos), random_state=42)
df_final = pd.concat([pos, neg]).sample(frac=1, random_state=42).reset_index(drop=True)

In [28]:
df_final['toxicity'].value_counts()

toxicity
0    7197
1    2399
Name: count, dtype: int64

In [29]:
df_final.head(10)

,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,key,cleaned_text,processed_text,polarization_list,pol_avg
0,2-7654,['11'],"kok bisa ya ada orang yang dukung israel kalo dulu kita dijajah belanda ada gak ya yang dukung belanda malah orang israel sendiri aja ada yg dukung palestina, coba liat palestina mana ada yg dukung israel, dr sini tau kan siapa yg salah🙏",Jewish,0,1,[1],0,0,0,1,0,2-7654_['11'],"Kenapa ada orang yang dukung Israel kalau dulu kita dijajah Belanda? Ada tidak ya yang dukung Belanda? Malah orang Israel sendiri ada yang dukung Palestina. Coba lihat, Palestina mana ada yang dukung Israel. Dari sini tahu kan siapa yang salah🙏","kenapa ada orang yang dukung israel kalau dulu kita dijajah belanda? ada tidak ya yang dukung belanda? malah orang israel sendiri ada yang dukung palestina. coba lihat, palestina mana ada yang dukung israel. dari sini tahu kan siapa yang salahfolded_hands",[1],1.000000
1,1-430,"['7', '15']","Stigma anti anies komunis itu senjatanya mereka untuk menakut2i pihak yang selama ini kritis pada anies jadi takut kritik anies,,modusnya mereka bisa dilihat secara kasat mata mereka senantiasa membabi buta melabeli komunis pihak yang kritis pada anies.☠️",Disabilitas,0,0,"[0, 0]",0,0,0,0,0,"1-430_['7', '15']",Stigma anti-Anies 'komunis' itu senjatanya; mereka gunakan itu untuk menakut-nakuti pihak yang selama ini kritis pada Anies sehingga takut mengkritik Anies. Modusnya bisa dilihat secara kasat mata: mereka senantiasa membabi buta melabeli komunis pihak yang kritis pada Anies. ☠️,stigma antianies komunis itu senjatanya mereka gunakan itu untuk menakutnakuti pihak yang selama ini kritis pada anies sehingga takut mengkritik anies. modusnya bisa dilihat secara kasat mata mereka senantiasa membabi buta melabeli komunis pihak yang kritis pada anies. skull_and_crossbones,"[0, 0]",0.000000
2,2-6367,"['7', '18']",Sentimen The Fed dan Data Ekspor-Impor China Tekan IHSG,Tionghoa,0,0,"[0, 0]",0,0,0,0,0,"2-6367_['7', '18']",Sentimen The Fed dan data ekspor-impor China tekan IHSG,sentimen the fed dan data eksporimpor china tekan ihsg,"[0, 0]",0.000000
3,2-5889,"['1', '18']","MEMBACA FREUD TENTANG ALAM BAWAH SADAR Refleksi pembiasaan bentuk pola sikap ... (DOKTRIN KONSERVATIF) Setiap manusia terbentuk oleh langkah jejak Alam keseharian turut mengikat Orang orang dekat membekali watak Darah nutrisi amat sangat mencipta harkat (DOKTRIN LIBERALISTIK) Logika empiris mengiringi detak nafas diri Bentuk pengalaman dan bangun pondasi persepsi Persepsi sikap dini respon pemula yang bersifat spontan dan mandiri Pengalaman masa pertumbuhan bentuk dasar peranggai dan pola sikap Tapak jejak perkembangan pengaruhi pola pikir dan peranggai diri Pengalaman yang membiasa panjang dan hiasi langkahnya, membentuk alam bawah sadar yang menjiwa kuat Itulah kerangka konsep diri sebagai dasar cara diri memposisikan harkat dan martabat Alam bawah sadar Situasi daya potensial yang menjiwa Selalu jadi pola pola prilaku hadapi setiap dinamika diri '𝗣𝘀𝘆𝗰𝗵𝗼𝗽𝗮𝘁𝗼𝗹𝗼𝗴𝘆 𝗼𝗳 𝗘𝘃𝗲𝗿𝘆𝗱𝗮𝘆 𝗟𝗶𝗳𝗲' 𝗞𝗮𝗿𝘆𝗮 𝗦𝗶𝗴𝗺𝘂𝗻𝗱 𝗙𝗿𝗲𝘂𝗱! Karya paling populer, Sigmund Freud menuangkan gagasan-gagasannya mengenai pengalaman bentuk alam bawah sadar Kealaman masa lalu, akan menjadi kesalahan atau kebenaran yang terjadi dalam kehidupan Jiwa yang terbiasa diisi ""a"" Akan membentuk prilaku serupa Buah akan jatuh di sekitar pohon Ikut menabur angin akan menubai badai Isilah alam bawah sadar dengan ilmu iman dan amal (DOKTRIN KRITIS ILMIYAH) Pembentuk alam bawah sadar menjadi pondasi karakter diri Akibat sentuhan kehidupan pribadi, kolega, keluarga, lingkup sosial dan pasien karyawannya Lewat pendekatan psikoanalisis, Freud menemukan betapa besar pengalaman bentuk alam bawah sadar Jejak pengalaman akan beri garis pemisah antara perilaku normal dan perilaku neurotik diri Neurotik ekspresi prilaku tidak normal, kehilangan jesadaran diri, dan ketidak mampuan diri jalani 

# BASELINE
epoch = 5
batch = 8
learning rate = 1e-5
weight_decay = 0.01

In [30]:
# def add_polarization_text(row):
#     return f"[POL={row['polarized']}] {row['processed_text']}"

# df_main["text_with_pol"] = df_main.apply(add_polarization_text, axis=1)

In [31]:
x = df_final[["processed_text", "pol_avg"]]
y = df_final['toxicity']

In [32]:
x_train_full, x_test, y_train_full, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.125, stratify=y_train_full, random_state=42
)

print("Stage 1 split sizes:", {"train": len(x_train), "val": len(x_val), "test": len(x_test)})

Stage 1 split sizes: {'train': 6716, 'val': 960, 'test': 1920}


In [33]:
class_weights_s1 = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights_s1 = torch.tensor(class_weights_s1, dtype=torch.float)
print("Stage 1 class weights (class 0, class 1):", class_weights_s1)

Stage 1 class weights (class 0, class 1): tensor([0.6667, 2.0000])


In [34]:
from datasets import Dataset, DatasetDict

def make_df(x, y):
    df = x.copy()
    df["label"] = y.values
    return df.reset_index(drop=True)

train_df = make_df(x_train, y_train)
val_df = make_df(x_val, y_val)
test_df = make_df(x_test, y_test)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['processed_text', 'pol_avg', 'label'],
        num_rows: 6716
    })
    validation: Dataset({
        features: ['processed_text', 'pol_avg', 'label'],
        num_rows: 960
    })
    test: Dataset({
        features: ['processed_text', 'pol_avg', 'label'],
        num_rows: 1920
    })
})


In [35]:
from transformers import (
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
TrainerCallback
)
tokenizer = AutoTokenizer.from_pretrained("indolem/indobertweet-base-uncased")

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# def tokenize(batch):
#     return tokenizer(batch["processed_text"], truncation=True, max_length=128, padding='max_length')

def tokenize_with_feature(batch):
    prefix = [f"Nilai rata-rata polarisasi: {v:.2f}" for v in batch["pol_avg"]]
    return tokenizer(prefix, batch["processed_text"], truncation="only_second", max_length=128, padding='max_length')

tokenized = dataset.map(tokenize_with_feature, batched=True)
tokenized = tokenized.remove_columns(["processed_text", "pol_avg"])

print(tokenizer.decode(tokenized["train"][0]["input_ids"]))

Map:   0%|          | 0/6716 [00:00<?, ? examples/s]

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

[CLS] nilai rata - rata polarisasi : 0. 00 [SEP] klasemen akhir asian games hangzhou indonesia finis posisi, china rekor medali emas. woman _ lifting _ weights cabang olahraga peraih medali untuk indonesia st _ place _ medal atlet peraih emas nd _ place _ medal atlet peraih perak rd _ place _ medal atlet peraih perunggu selengkapnya [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [37]:
tokenized_datasets_1 = dataset.map(tokenize_with_feature, batched=True)

tokenized_datasets_1 = tokenized_datasets_1.remove_columns(["processed_text",'pol_avg'])
tokenized_datasets_1 = tokenized_datasets_1.rename_column("label", "labels")
tokenized_datasets_1.set_format("torch")

train_dataset_1 = tokenized_datasets_1["train"]
val_dataset_1 = tokenized_datasets_1["validation"]
test_dataset_1 = tokenized_datasets_1["test"]

Map:   0%|          | 0/6716 [00:00<?, ? examples/s]

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

In [38]:
import numpy as np
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
)

def compute_metrics_s1(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=1)[:, 1]
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs),
    }

In [39]:
# from transformers import AutoModelForSequenceClassification

# model_s1 = AutoModelForSequenceClassification.from_pretrained(
#     "indolem/indobertweet-base-uncased",
#     num_labels=2
# )

In [40]:
# Trainer that applies class weights to CrossEntropyLoss, used to handle label
# imbalance in Stage 1.
class WeightedTrainerSingleLabel(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = nn.CrossEntropyLoss(weight=weight)
        # logits.size(-1) instead of model.config.num_labels: stays correct even if
        # Trainer wraps the model in nn.DataParallel (which doesn't expose .config).
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [41]:
class OptunaPruningCallback(TrainerCallback):
    def __init__(self, trial, metric="eval_f1"):
        self.trial = trial
        self.metric = metric

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics or self.metric not in metrics:
            return
        self.trial.report(metrics[self.metric], step=int(round(state.epoch)))
        if self.trial.should_prune():
            raise optuna.TrialPruned()

In [ ]:
from transformers import set_seed

MODEL_NAME = "indolem/indobertweet-base-uncased"
N_TRIALS = 25
OUTPUT_ROOT = "./optuna_s1"

def freeze_bottom(model, n_freeze, freeze_emb):
    """Bekukan embeddings (opsional) + n_freeze layer encoder terbawah.
    Dipakai di objective dan final training supaya setup-nya pasti sama."""
    if freeze_emb:
        for p in model.bert.embeddings.parameters():
            p.requires_grad = False
    for layer in model.bert.encoder.layer[:n_freeze]:
        for p in layer.parameters():
            p.requires_grad = False

def objective_s1(trial):
    # Prioritas tinggi
    lr = trial.suggest_float("learning_rate", 8e-6, 6e-5, log=True)
    warmup = trial.suggest_float("warmup_ratio", 0.0, 0.2)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    epochs = trial.suggest_int("num_train_epochs", 3, 8)
    cw_alpha = trial.suggest_float("class_weight_alpha", 0.3, 1.0)
    # Prioritas menengah
    dropout = trial.suggest_float("dropout", 0.1, 0.3)
    # llrd_decay = trial.suggest_float("llrd_decay", 0.75, 1.0)
    scheduler = trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine"])
    # Prioritas rendah
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    n_freeze = trial.suggest_int("n_frozen_layers", 0, 6)
    freeze_emb = trial.suggest_categorical("freeze_embeddings", [False, True])

    out_dir = f"{OUTPUT_ROOT}/trial_{trial.number}"
    trainer, model = None, None
    try:
        # set_seed mengunci random, numpy, dan torch -> inisialisasi classifier ikut reproducible
        set_seed(42)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=2,
            hidden_dropout_prob=dropout,
            attention_probs_dropout_prob=dropout,
        )
        freeze_bottom(model, n_freeze, freeze_emb)

        # alpha=1 -> "balanced" penuh, alpha<1 -> bobot lebih lunak
        class_weights = class_weights_s1 ** cw_alpha
        n_gpu = max(1, torch.cuda.device_count())
        steps_per_epoch = math.ceil(len(train_dataset_1) / (batch_size * n_gpu))
        warmup_steps = int(warmup * steps_per_epoch * epochs)

        args = TrainingArguments(
            output_dir=out_dir,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=64,
            learning_rate=lr,
            warmup_steps=warmup_steps,
            weight_decay=weight_decay,
            lr_scheduler_type=scheduler,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            save_only_model=True,
            save_total_limit=1,
            metric_for_best_model="f1",
            greater_is_better=True,
            fp16=torch.cuda.is_available(),
            logging_steps=50,
            disable_tqdm=True,
            report_to=[],
            seed=42,
        )

        trainer = WeightedTrainerSingleLabel(
            model=model,
            args=args,
            train_dataset=train_dataset_1,
            eval_dataset=val_dataset_1,
            compute_metrics=compute_metrics_s1,
            class_weights=class_weights,
            callbacks=[
                EarlyStoppingCallback(early_stopping_patience=2),
                OptunaPruningCallback(trial, metric="eval_f1"),
            ],
        )
        trainer.train()
        return trainer.state.best_metric
    finally:
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()
        shutil.rmtree(out_dir, ignore_errors=True)

In [ ]:
study_s1 = optuna.create_study(
    direction="maximize",
    study_name="stage1_balance_polarized_v2",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),  # jangan prune di epoch 1, F1 masih dipengaruhi warmup
    storage="sqlite:///optuna_s1.db",
    load_if_exists=True,
)
study_s1.optimize(objective_s1, n_trials=N_TRIALS)

[I 2026-10-12 05:48:46,043] A new study created in RDB with name: stage1_balance_polarized_v2


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  445MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3494', 'grad_norm': '4.461', 'learning_rate': '1.596e-05', 'epoch': '2.143'}
{'loss': '0.3716', 'grad_norm': '4.354', 'learning_rate': '1.497e-05', 'epoch': '2.381'}
{'loss': '0.3288', 'grad_norm': '4.98', 'learning_rate': '1.399e-05', 'epoch': '2.619'}
{'loss': '0.3528', 'grad_norm': '6.878', 'learning_rate': '1.3e-05', 'epoch': '2.857'}
{'eval_loss': '0.3896', 'eval_accuracy': '0.8344', 'eval_precision': '0.6239', 'eval_recall': '0.85', 'eval_f1': '0.7196', 'eval_f1_macro': '0.801', 'eval_roc_auc': '0.9058', 'eval_pr_auc': '0.7732', 'eval_runtime': '4.173', 'eval_samples_per_second': '230.1', 'eval_steps_per_second': '1.917', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3453', 'grad_norm': '6.736', 'learning_rate': '1.202e-05', 'epoch': '3.095'}
{'loss': '0.303', 'grad_norm': '9.111', 'learning_rate': '1.104e-05', 'epoch': '3.333'}
{'loss': '0.2897', 'grad_norm': '5.359', 'learning_rate': '1.005e-05', 'epoch': '3.571'}
{'loss': '0.3276', 'grad_norm': '5.494', 'learning_rate': '9.069e-06', 'epoch': '3.81'}
{'eval_loss': '0.3851', 'eval_accuracy': '0.8562', 'eval_precision': '0.6932', 'eval_recall': '0.7625', 'eval_f1': '0.7262', 'eval_f1_macro': '0.8144', 'eval_roc_auc': '0.9046', 'eval_pr_auc': '0.7753', 'eval_runtime': '4.17', 'eval_samples_per_second': '230.2', 'eval_steps_per_second': '1.918', 'epoch': '4'}


[I 2026-10-12 08:24:41,173] Trial 24 pruned. 


In [44]:
print("Best Stage 1 params:", study_s1.best_params)
print("Best Stage 1 validation F1:", study_s1.best_value)

Best Stage 1 params: {'learning_rate': 3.609736905213189e-05, 'warmup_ratio': 0.011902006240887913, 'batch_size': 16, 'num_train_epochs': 5, 'class_weight_alpha': 0.3668117043064722, 'dropout': 0.13472074560845987, 'lr_scheduler_type': 'linear', 'weight_decay': 0.04842130259776237, 'n_frozen_layers': 5, 'freeze_embeddings': True}
Best Stage 1 validation F1: 0.7341269841269841


In [48]:
best_s1 = study_s1.best_params
class_weights = class_weights_s1 ** best_s1["class_weight_alpha"]

n_gpu = max(1, torch.cuda.device_count())
steps_per_epoch = math.ceil(len(train_dataset_1) / (best_s1["batch_size"] * n_gpu))
warmup_steps = int(best_s1["warmup_ratio"] * steps_per_epoch * best_s1["num_train_epochs"])

set_seed(42)
model_s1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    hidden_dropout_prob=best_s1["dropout"],
    attention_probs_dropout_prob=best_s1["dropout"],
)

freeze_bottom(model_s1, best_s1["n_frozen_layers"], best_s1["freeze_embeddings"])

args_s1 = TrainingArguments(
    output_dir="./best_s1",
    num_train_epochs=best_s1["num_train_epochs"],
    per_device_train_batch_size=best_s1["batch_size"],
    per_device_eval_batch_size=64,
    learning_rate=best_s1["learning_rate"],
    weight_decay=best_s1["weight_decay"],
    warmup_steps=warmup_steps,
    lr_scheduler_type=best_s1["lr_scheduler_type"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    save_only_model=True,
    logging_steps=50,
    disable_tqdm=False,
    report_to=[],
    seed=42,
)

trainer_s1 = WeightedTrainerSingleLabel(
    model=model_s1,
    args=args_s1,
    train_dataset=train_dataset_1,
    eval_dataset=val_dataset_1,
    compute_metrics=compute_metrics_s1,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer_s1.train()
print(trainer_s1.evaluate())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Macro,Roc Auc,Pr Auc
1,0.397815,0.447059,0.787500,0.546392,0.883333,0.675159,0.758632,0.899832,0.766735
2,0.368347,0.371399,0.848958,0.659933,0.816667,0.729981,0.812568,0.903999,0.768722
3,0.324862,0.386120,0.848958,0.657807,0.825000,0.731978,0.813415,0.905880,0.766527
4,0.296057,0.391349,0.860417,0.700758,0.770833,0.734127,0.819747,0.906580,0.767342
5,0.230901,0.406928,0.856250,0.694656,0.758333,0.725100,0.813890,0.907228,0.771433


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,F1 Macro,Roc Auc,Pr Auc
0.230901,0.391349,5,0.860417,0.700758,0.770833,0.734127,0.819747,0.906580,0.767342


{'eval_loss': 0.3913491666316986, 'eval_accuracy': 0.8604166666666667, 'eval_precision': 0.7007575757575758, 'eval_recall': 0.7708333333333334, 'eval_f1': 0.7341269841269841, 'eval_f1_macro': 0.819747107882701, 'eval_roc_auc': 0.9065798611111111, 'eval_pr_auc': 0.7673417150715535}


In [49]:
from sklearn.metrics import classification_report

In [50]:
pred1 = trainer_s1.predict(test_dataset_1)
pred_label1 = pred1.predictions.argmax(axis=1)

probs1_test = torch.softmax(torch.tensor(pred1.predictions), dim=-1)[:, 1].numpy()

print("Stage 1 classification report -- test set:")
print(classification_report(y_test, pred_label1))

Stage 1 classification report -- test set:
              precision    recall  f1-score   support

           0       0.90      0.89      0.90      1440
           1       0.68      0.72      0.70       480

    accuracy                           0.85      1920
   macro avg       0.79      0.80      0.80      1920
weighted avg       0.85      0.85      0.85      1920

